# Earth–Sun distance correction — from the ephemeris table to adjusted radiance

EMIT **L1B radiance is not Earth–Sun-distance normalized** (L2A reflectance is), so it carries the seasonal $1/d^2$ solar-flux term — about **±3.4%** across the year (perihelion $\approx0.983$ AU early January, aphelion $\approx1.017$ AU early July). Over a pseudo-invariant site, an imbalanced matched-pair sample lets that *periodic* term masquerade as radiometric *drift*.

This notebook does two things:

1. **Derives the correction from the ephemeris table** — `config/earth_sun_distance.txt`, the pipeline default (`radiometry.e0_method: table`) — with the governing equations: how a tabulated distance $d$ becomes a per-scene, then per-pair, radiance adjustment.
2. Produces **one clean figure**: unadjusted radiance, Earth–Sun-adjusted radiance, and reflectance drift spectra as solid lines.

It also **audits** the built-in pipeline correction: every step here uses the *same* `emit_pics.earth_sun` provider the pipeline uses (selected by `run_config.yaml`), and the equivalence check confirms this ratio-domain correction reproduces the pipeline's per-scene division to machine precision.

> **No double-correction.** The loaded aggregated delta tables may already be E0-normalized (if produced with the correction on). The cells below read the provenance sidecar's `earth_sun_normalize` flag and reconstruct the **raw** radiance delta first when needed, so the before/after is always meaningful and never corrected twice.

**Companion:** `sza_leak_diagnostic.ipynb` (the solar-zenith illumination term).

In [ ]:
# Repo-relative bootstrap: find the repo root (holds run_config.yaml) and import.
import sys, pathlib, json, glob
_p = pathlib.Path.cwd()
while not (_p / "run_config.yaml").exists() and _p != _p.parent:
    _p = _p.parent
if str(_p) not in sys.path:
    sys.path.insert(0, str(_p))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import kendalltau, theilslopes

from emit_pics import paths
from emit_pics.run_config import load_config
from emit_pics.earth_sun import (
    earth_sun_distance_au, table_E0, spencer_E0, resolve_e0)
from emit_pics.matched_pairs_drift_analysis import (
    load_aggregated_delta_tables, compute_drift_spectrum, DAYS_PER_YEAR)

cfg   = load_config()
ALPHA = cfg.alpha
SIZE  = cfg.patch_size
E0_FN = resolve_e0(cfg.e0_method)   # the SAME E0 provider the pipeline used (default: table)

# Pool the aggregated (one-row-per-pair) delta tables across all PICS of this size.
df = load_aggregated_delta_tables(paths.DELTA_AGG_DIR, SIZE)
if df.empty:
    raise SystemExit(
        f"No aggregated delta tables for size {SIZE!r} in {paths.DELTA_AGG_DIR}.\n"
        "Run the analyze stage (scripts/run_PICS_allpatches_aggregate_pairs.py) first.")

n_pairs = df.groupby(["PICS_ID", "pair_id"]).ngroups
print(f"patch size {SIZE} km | {len(df)} rows | {df['PICS_ID'].nunique()} PICS | "
      f"{n_pairs} unique pairs | {df['band_index'].nunique()} channels")
print(f"E0 provider (run_config.e0_method) : {cfg.e0_method}")


def _delta_earth_sun_flag(agg_dir, size):
    """Read one delta sidecar to learn if the loaded radiance deltas already have
    E0 removed. Legacy tables (no flag in the sidecar) predate the toggle and are
    therefore raw."""
    for f in sorted(glob.glob(str(pathlib.Path(agg_dir) / f"*_{size}_delta.csv"))):
        sc = pathlib.Path(f + ".json")
        if sc.exists():
            rc = json.loads(sc.read_text()).get("run_config", {})
            if "earth_sun_normalize" in rc:
                return bool(rc["earth_sun_normalize"])
    return False

STORED_NORM = _delta_earth_sun_flag(paths.DELTA_AGG_DIR, SIZE)
print(f"config earth_sun_normalize = {cfg.earth_sun_normalize}   |   "
      f"loaded radiance deltas are "
      f"{'E0-NORMALIZED (pipeline)' if STORED_NORM else 'RAW (un-normalized)'}")

In [ ]:
# --- Adjustable plot y-limits -------------------------------------------------
# The drift-spectrum panels otherwise autoscale to the most extreme (often
# numerically unstable) channel, which flattens the ~0.5 %/yr differences between
# the series. Set tight limits here; use None to autoscale. Re-run the plot cells
# after changing these.
SPECTRUM_YLIM = (-1.5, 1.5)   # drift-spectrum panels, %/yr
DIAG_YLIM     = None          # signed-ΔSZA / flux-ratio scatter (degrees or %)

## 1. From the distance table to adjusted radiance

The pipeline's default provider (`radiometry.e0_method: table`) is an **ephemeris lookup table**, `config/earth_sun_distance.txt`, giving the Earth–Sun distance $d$ (AU) for each day-of-year:

| day-of-year | $d$ (AU) | phase |
| --- | --- | --- |
| ~3 (early Jan) | 0.9833 | perihelion (closest) |
| ~184 (early Jul) | 1.0167 | aphelion (farthest) |

**Symbols** (per scene, per channel):

| symbol | meaning |
| --- | --- |
| $L_\text{meas}$ | the **measured at-sensor radiance** EMIT L1B actually records for the scene, at its real Earth–Sun distance $d$ (units µW cm⁻² sr⁻¹ nm⁻¹; this is the value in the `*_perchannel_rad.csv` stats) |
| $L_{1\,\mathrm{AU}}$ | the **distance-normalized radiance** — what the *same* surface and viewing/illumination geometry would give at the mean Earth–Sun distance ($d = 1$ AU); this is the quantity we want |
| $d$ | Earth–Sun distance (AU) on the acquisition date, from the table |
| $d_0$ | mean Earth–Sun distance $= 1$ AU |
| $E_0 \equiv (d_0/d)^2 = 1/d^2$ | the eccentricity (distance) factor |

**Inverse-square law.** Top-of-atmosphere solar irradiance falls off as $1/d^2$, and at-sensor radiance over a fixed surface scales with the incident irradiance, so a scene acquired at distance $d$ measures

$$L_\text{meas}(d) = \frac{L_{1\,\mathrm{AU}}}{d^{2}}.$$

L2A reflectance already divides this out; L1B radiance does not. Normalizing a scene to 1 AU is therefore a single per-scene scalar:

$$\boxed{\,L_{1\,\mathrm{AU}} = L_\text{meas}\,d^{2} = \dfrac{L_\text{meas}}{E_0}\,}$$

$E_0$ is spatially and spectrally uniform over a scene, so dividing the per-channel **median** (or mean) radiance by $E_0$ is *exact* — identical to dividing every pixel. (The closed-form Spencer-1971 series, `e0_method: spencer`, gives the same $E_0$ to <0.1%.)

In [ ]:
# Per-scene Earth–Sun distance d (AU) and E0 from the CONFIGURED provider, so the
# notebook matches whatever the pipeline used (default: the ephemeris table).
df = df.copy()
df["d_t1"]  = earth_sun_distance_au(df["t1_dt"])
df["d_t2"]  = earth_sun_distance_au(df["t2_dt"])
df["E0_t1"] = E0_FN(df["t1_dt"])
df["E0_t2"] = E0_FN(df["t2_dt"])

pairs = df.drop_duplicates(subset=["PICS_ID", "pair_id"]).copy()
pairs["flux_ratio_pct"] = 100.0 * (pairs["E0_t2"] / pairs["E0_t1"] - 1.0)

print(f"E0 provider              : {cfg.e0_method}")
print(f"d range across scenes     : [{df['d_t1'].min():.4f}, {df['d_t1'].max():.4f}] AU")
print(f"E0 = 1/d^2 range          : [{df['E0_t1'].min():.4f}, {df['E0_t1'].max():.4f}]  "
      f"(annual span ~{100*(df['E0_t1'].max()/df['E0_t1'].min()-1):.1f}%)")

# --- Worked example: the pair with the largest distance contrast ---------------
ex = pairs.loc[pairs["flux_ratio_pct"].abs().idxmax()]
factor = (ex["d_t2"] / ex["d_t1"]) ** 2          # the per-pair adjustment (d2/d1)^2
print("\nWorked example (largest distance contrast in the pool):")
print(f"  t1 = {ex['t1_dt']}   d1 = {ex['d_t1']:.4f} AU   ->  E0_1 = {ex['E0_t1']:.4f}")
print(f"  t2 = {ex['t2_dt']}   d2 = {ex['d_t2']:.4f} AU   ->  E0_2 = {ex['E0_t2']:.4f}")
print(f"  adjustment factor (d2/d1)^2 = {factor:.4f}  "
      f"=>  each measured t1->t2 radiance ratio is scaled by {100*(factor-1):+.2f}%")

## 2. The matched-pair equation — why the term can look like drift

**Symbols** (per matched pair, per channel; all radiance symbols as defined in §1):

| symbol | meaning |
| --- | --- |
| $t_1,\; t_2$ | the earlier / later acquisition of the matched pair |
| $\Delta t = t_2 - t_1$ | the time gap between them (years) |
| $\Delta R_\text{raw} = 100\left(\dfrac{L_\text{meas}(t_2)}{L_\text{meas}(t_1)} - 1\right)$ | the **measured** percent change in radiance from $t_1$ to $t_2$ (the `delta_rad_pct_median` column) |
| $\Delta R_\text{adj}$ | the same percent change after both scenes are distance-normalized to 1 AU |

Substituting $L_\text{meas} = L_{1\,\mathrm{AU}}/d^{2}$ into $\Delta R_\text{raw}$ and rearranging, the distance-**adjusted** change is

$$1 + \frac{\Delta R_\text{adj}}{100} \;=\; \Bigl(1 + \frac{\Delta R_\text{raw}}{100}\Bigr)\left(\frac{d(t_2)}{d(t_1)}\right)^{2} \;=\; \frac{1 + \Delta R_\text{raw}/100}{E_0(t_2)/E_0(t_1)}.$$

So the entire correction is the achromatic per-pair factor $\bigl(d(t_2)/d(t_1)\bigr)^2$. The drift rate is the slope of $\Delta R$ against $\Delta t$, so this factor changes the drift **only if the flux ratio $E_0(t_2)/E_0(t_1)$ is itself correlated with $\Delta t$** — which happens when the pair sample is seasonally imbalanced across baselines. The scatter below quantifies exactly that correlation.

In [ ]:
m = pairs["flux_ratio_pct"].notna() & pairs["dt_days"].notna()
x = pairs.loc[m, "dt_days"].to_numpy()
y = pairs.loc[m, "flux_ratio_pct"].to_numpy()
tau, p = kendalltau(x, y)
slope, intc, lo, hi = theilslopes(y, x)
print(f"flux ratio (%) vs Δt :  Theil–Sen slope {slope * DAYS_PER_YEAR:+.4f} %/yr   "
      f"Kendall τ = {tau:+.3f}   p = {p:.2g}")

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].scatter(x, y, s=12, alpha=0.5)
xr = np.array([x.min(), x.max()])
ax[0].plot(xr, intc + slope * xr, "k-", lw=1.5)
ax[0].axhline(0.0, color="0.6", lw=0.8)
ax[0].set_xlabel("Δt (days)")
ax[0].set_ylabel("100·(E0(t2)/E0(t1) − 1)  [%]")
ax[0].set_title(f"Earth–Sun flux ratio vs Δt   (τ={tau:+.2f}, p={p:.2g})")
if DIAG_YLIM is not None:
    ax[0].set_ylim(DIAG_YLIM)
ax[1].hist(y, bins=40, color="tab:purple", alpha=0.85)
ax[1].axvline(0.0, color="k", lw=1.0)
ax[1].axvline(float(np.mean(y)), color="tab:red", lw=1.5, ls="--",
             label=f"mean {np.mean(y):+.2f}%")
ax[1].set_xlabel("flux ratio [%]")
ax[1].set_ylabel("pairs")
ax[1].set_title("distribution of Earth–Sun flux ratio")
ax[1].legend()
plt.tight_layout()
plt.show()

## 3. Apply the correction and audit against the pipeline

Using the per-pair factor from §2, form the **adjusted** radiance change from the **raw** one and refit the drift. Because the loaded tables may *already* be E0-normalized by the pipeline, the raw series is reconstructed first (via the inverse factor) so nothing is double-corrected. When the tables are already normalized, this manual, ratio-domain result must reproduce the stored pipeline values — the `AUDIT` line confirms it to machine precision, proving the notebook math and the pipeline's per-scene $L/E_0$ division are the same operation.

In [ ]:
def remove_illumination(delta_pct, ratio):
    # Adjusted % change = raw scaled by the inverse of the illumination ratio.
    # ratio = illum(t2)/illum(t1); (1 + dR/100) -> (1 + dR/100) / ratio.
    delta_pct = np.asarray(delta_pct, dtype=float)
    ratio     = np.asarray(ratio, dtype=float)
    return 100.0 * ((1.0 + delta_pct / 100.0) / ratio - 1.0)


def apply_illumination(delta_pct, ratio):
    # Inverse of remove_illumination: re-impose `ratio`. Used to reconstruct the
    # RAW radiance delta from an already-E0-normalized one (never double-correct).
    delta_pct = np.asarray(delta_pct, dtype=float)
    ratio     = np.asarray(ratio, dtype=float)
    return 100.0 * ((1.0 + delta_pct / 100.0) * ratio - 1.0)


def median_drift(spec):
    return float(np.nanmedian(spec["slope_pct_yr"]))


def _absorption_runs(spec):
    """(lo, hi) wavelength spans of contiguous water-vapor absorption bands."""
    s = spec.sort_values("wavelength")
    wl = s["wavelength"].to_numpy()
    ab = s["absorption_band"].fillna(False).to_numpy().astype(bool)
    runs, i, n = [], 0, len(ab)
    while i < n:
        if ab[i]:
            j = i
            while j + 1 < n and ab[j + 1]:
                j += 1
            runs.append((wl[i], wl[j])); i = j + 1
        else:
            i += 1
    return runs


def plot_drift_lines(specs_labels_colors, title, *, ylim=None,
                     absorption_from=None, save=None):
    """One clean figure: each drift spectrum as a SOLID line vs wavelength.

    ``slope_pct_yr`` is NaN in absorption / unfit channels, so the line breaks
    there (no bridging across gaps). Water-vapor bands are lightly shaded.
    """
    fig, ax = plt.subplots(figsize=(11, 5))
    if absorption_from is not None:
        for k, (lo, hi) in enumerate(_absorption_runs(absorption_from)):
            ax.axvspan(lo, hi, color="0.9", zorder=0,
                       label="water-vapor band" if k == 0 else None)
    for spec, lab, c in specs_labels_colors:
        s = spec.sort_values("wavelength")
        ax.plot(s["wavelength"], s["slope_pct_yr"], "-", lw=2.0, color=c,
                label=lab, zorder=3)
    ax.axhline(0.0, color="k", lw=0.8, zorder=2)
    if ylim is not None:
        ax.set_ylim(ylim)
    ax.set_xlabel("wavelength (nm)")
    ax.set_ylabel("radiometric drift (%/yr)")
    ax.set_title(title)
    ax.grid(True, alpha=0.25)
    ax.legend(frameon=True, framealpha=0.9, loc="best")
    fig.tight_layout()
    if save is not None:
        fig.savefig(save, dpi=300, bbox_inches="tight")
        print(f"saved figure -> {save}")
    plt.show()

In [ ]:
f_es = (df["E0_t2"] / df["E0_t1"]).to_numpy()

# Reconstruct the RAW radiance delta whatever is stored, then apply the pipeline's
# E0 correction to it. If the loaded tables are already normalized, undo it first
# (apply_illumination) so nothing is double-corrected.
stored     = df["delta_rad_pct_median"].to_numpy()
delta_raw  = apply_illumination(stored, f_es) if STORED_NORM else stored
delta_corr = remove_illumination(delta_raw, f_es)     # == the pipeline's per-scene E0 division

df_raw  = df.copy(); df_raw["delta_rad_pct_median"]  = delta_raw
df_corr = df.copy(); df_corr["delta_rad_pct_median"] = delta_corr

spec_rad      = compute_drift_spectrum(df_raw,  "rad", alpha=ALPHA)   # before (raw)
spec_rad_corr = compute_drift_spectrum(df_corr, "rad", alpha=ALPHA)   # after  (E0-corrected)
spec_rfl      = compute_drift_spectrum(df,      "rfl", alpha=ALPHA)   # reflectance reference

print(f"median radiance drift  raw .............. {median_drift(spec_rad):+.3f} %/yr")
print(f"median radiance drift  E0-corrected ..... {median_drift(spec_rad_corr):+.3f} %/yr")
print(f"median reflectance drift (reference) .... {median_drift(spec_rfl):+.3f} %/yr")

# AUDIT: when the pipeline already corrected, the manual correction must reproduce it.
if STORED_NORM:
    dmax = float(np.nanmax(np.abs(delta_corr - stored)))
    print(f"\nAUDIT  max |manual − pipeline| radiance delta = {dmax:.2e} %  "
          f"-> {'PASS (equivalent)' if dmax < 1e-6 else 'CHECK — investigate'}")
else:
    print("\n(loaded tables are raw; 'E0-corrected' above is what the pipeline "
          "produces with radiometry.earth_sun_normalize: true)")

In [ ]:
# === THE figure: unadjusted RAD, Earth–Sun-adjusted RAD, reflectance (solid lines) ===
SUPP_FIG = pathlib.Path(_p) / "figures" / "earth_sun_correction_drift_spectrum.png"
SUPP_FIG.parent.mkdir(parents=True, exist_ok=True)

plot_drift_lines(
    [(spec_rad,      "radiance — unadjusted (raw L1B)", "tab:red"),
     (spec_rad_corr, "radiance — Earth–Sun adjusted",  "tab:blue"),
     (spec_rfl,      "reflectance (L2A reference)",     "tab:green")],
    f"EMIT drift spectrum — Earth–Sun distance adjustment (E0 via {cfg.e0_method})",
    ylim=SPECTRUM_YLIM, absorption_from=spec_rfl, save=SUPP_FIG)

## Appendix — combined cos(SZA)·E0 illumination correction *(optional)*

The full first-order illumination factor is the **product** of the distance term and the solar-zenith term. This removes both at once — the closest matched-pairs analog to what the pooled BRDF model does explicitly — as a fourth line for reference. It is not part of the main result above.

In [ ]:
# Combined cos(SZA)·E0, applied to the reconstructed RAW radiance delta (so this
# is raw -> full illumination-adjusted, independent of what the tables stored).
f_both = ((np.cos(np.radians(df["SZA_t2"])) / np.cos(np.radians(df["SZA_t1"])))
          * (df["E0_t2"] / df["E0_t1"])).to_numpy()

df_both = df.copy()
df_both["delta_rad_pct_median"] = remove_illumination(delta_raw, f_both)
spec_rad_both = compute_drift_spectrum(df_both, "rad", alpha=ALPHA)

print(f"median radiance drift  cos(SZA)·E0-adjusted .. {median_drift(spec_rad_both):+.3f} %/yr")
plot_drift_lines(
    [(spec_rad,      "radiance — unadjusted (raw L1B)",   "tab:red"),
     (spec_rad_corr, "radiance — Earth–Sun adjusted",     "tab:blue"),
     (spec_rad_both, "radiance — cos(SZA)·E0 adjusted",   "tab:orange"),
     (spec_rfl,      "reflectance (L2A reference)",       "tab:green")],
    "Appendix: raw vs Earth–Sun vs full cos(SZA)·E0 illumination adjustment",
    ylim=SPECTRUM_YLIM, absorption_from=spec_rfl)

## How to read this
- **The figure in §3** overlays the unadjusted radiance, Earth–Sun-adjusted radiance, and reflectance drift spectra. The adjustment is **achromatic** — the same per-pair $(d_2/d_1)^2$ factor at every wavelength — so it shifts the whole radiance curve by a near-constant vertical offset. That flat offset is the fingerprint of an illumination/sampling term; a spectrally-structured *sensor* effect could not produce it.
- **Adjusted radiance collapses toward reflectance** ⇒ the seasonal $1/d^2$ term was a real leak in the raw radiance drift (reflectance never carried it, since L2A removes $E_0$). Because the correction is derived from acquisition **dates only** — blind to the radiance values and to any drift — that convergence is genuine validation, not circularity.
- **AUDIT PASS** confirms this ratio-domain correction is numerically identical to the pipeline's read-time per-scene division (`radiometry.earth_sun_normalize`, provider `e0_method`).
- Any drift remaining in the adjusted radiance is the residual **angular** illumination term ($\cos\text{SZA}$ / BRDF) or genuine sensor behaviour — see the appendix and, definitively, the pooled BRDF fit.

**Tip:** to reproduce the contrast from truly raw tables, regenerate `analyze` with `radiometry.earth_sun_normalize: false`; this notebook then rebuilds, from those raw tables, exactly the adjusted spectrum the pipeline produces with the correction on.